## CMAPSS Predictive Maintenance Pipeline
### Gold Layer

In [0]:
from pyspark.sql.functions import(
  col, avg, max as spark_max, min as spark_min, 
  count, stddev, round as spark_round,
  when, lit, percentile_approx
)

silver_train = spark.read.format("delta").table("cmapss_project.silver.silver_train")
silver_test = spark.read.format("delta").table("cmapss_project.silver.silver_test")

KEY_SENSORS = ["sensor_2", "sensor_3", "sensor_4", "sensor_7",
               "sensor_11", "sensor_12", "sensor_15", "sensor_17",
               "sensor_20", "sensor_21"]

DATASETS = ["FD001", "FD002", "FD003", "FD004"]

print(f"silver_train rows: {silver_train.count()}")
print(f"silver_test rows:  {silver_test.count()}")

silver_train rows: 160359
silver_test rows:  104897


### Gold Table 1: Fleet Health Summary

In [0]:
gold_fleet_health = (
  silver_train
  .groupBy("unit_id", "dataset_id", "data_split")
  .agg(
    spark_max("cycle").alias("total_cycles"),
    spark_min("rul").alias("min_rul"),
    spark_max("rul").alias("max_rul"),
    avg("sensor_2").alias("avg_sensor_2"),
    avg("sensor_7").alias("avg_sensor_7"),
    avg("sensor_11").alias("avg_sensor_11"),
    avg("sensor_12").alias("avg_sensor_12")
  )

  # Add fault mode label based on dataset
  .withColumn(
    "fault_mode",
    when(col("dataset_id").isin("FDOO1", "FD002"), "HPC Degradation")
    .otherwise("HPC + Fan Degradation")
  )

  # Add operating condition label
  .withColumn(
    "operating_conditions",
    when(col("dataset_id").isin("FD001", "FDOO3"), "Single (Sea Level)")
    .otherwise("Multiple (6 Conditions)")

  ).orderBy("dataset_id", "unit_id")

)

(
  gold_fleet_health.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("cmapss_project.gold.gold_fleet_health")
)







### Gold Table 2: Sensor Degradation Trends

In [0]:
"""Average sensor readings bucketed by RUL range"""

# Create RUL buckets
silver_train_bucketed = silver_train.withColumn(
  "rul_bucket",
  when(col("rul") <= 25, "0-25 (Critical)")
  .when(col("rul") <= 50, "26-50 (Warning)")
  .when(col("rul") <= 100, "51-100 (Moderate)")
  .when(col("rul") <= 150, "101-150 (Healthy)")
  .otherwise("150+ (New)") 
)

# Aggregate average sensor reading per bucket per dataset
gold_sensor_trends = (
    silver_train_bucketed
    .groupBy("dataset_id", "rul_bucket")
    .agg(
        count("*").alias("row_count"),
        *[spark_round(avg(s), 4).alias(f"avg_{s}") for s in KEY_SENSORS]
    )
    .orderBy("dataset_id", "rul_bucket")
)

(
    gold_sensor_trends.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("cmapss_project.gold.gold_sensor_trends")
)

### Gold Table 3: Operating Conditions Analysis


In [0]:
# Breakdown of op_settings and thei effect on engine lifespan
gold_operating_conditions = (
    silver_train
    .groupBy("dataset_id", 
             spark_round("op_setting_1", 1).alias("op_setting_1_rounded"),
             spark_round("op_setting_2", 1).alias("op_setting_2_rounded"))
    .agg(
        count("unit_id").alias("observation_count"),
        spark_round(avg("rul"), 2).alias("avg_rul"),
        spark_round(stddev("rul"), 2).alias("std_rul"),
        spark_round(avg("sensor_2"), 4).alias("avg_sensor_2"),
        spark_round(avg("sensor_7"), 4).alias("avg_sensor_7"),
        spark_round(avg("sensor_11"), 4).alias("avg_sensor_11")
    )
    .orderBy("dataset_id", "avg_rul")
)

(
    gold_operating_conditions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("cmapss_project.gold.gold_operating_conditions")
)

### Gold Table 4: Fault Mode Comparison

In [0]:
# Compare single vs dual fault mode engines

# Lifespan per engine
engine_lifespan = (
    silver_train
    .groupBy("unit_id", "dataset_id")
    .agg(spark_max("cycle").alias("total_cycles"))
)

gold_fault_comparison = (
    engine_lifespan
    .withColumn(
        "fault_mode",
        when(col("dataset_id").isin("FD001", "FD002"), "Single Fault")
        .otherwise("Dual Fault")
    )
    .withColumn(
      "condition_type",
      when(col("dataset_id").isin("FD001", "FD003"), "Single Condition")
      .otherwise("Multi Condition")
    )
    .groupBy("dataset_id", "fault_mode", "condition_type")
    .agg(
        count("unit_id").alias("engine_count"),
        spark_round(avg("total_cycles"), 1).alias("avg_lifespan"),
        spark_round(stddev("total_cycles"), 1).alias("std_lifespan"),
        spark_min("total_cycles").alias("min_lifespan"),
        spark_max("total_cycles").alias("max_lifespan"),
        spark_round(percentile_approx("total_cycles", 0.25), 1).alias("p25_lifespan"),
        spark_round(percentile_approx("total_cycles", 0.75), 1).alias("p75_lifespan")
    )
    .orderBy("dataset_id")
)

(
    gold_fault_comparison.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("cmapss_project.gold.gold_fault_comparison")
)


In [0]:
display(gold_sensor_trends)

dataset_id,rul_bucket,row_count,avg_sensor_2,avg_sensor_3,avg_sensor_4,avg_sensor_7,avg_sensor_11,avg_sensor_12,avg_sensor_15,avg_sensor_17,avg_sensor_20,avg_sensor_21
FD001,0-25 (Critical),2600,0.6663,0.6165,0.6998,0.34,0.6794,0.3252,0.6782,0.6255,0.3111,0.3152
FD001,101-150 (Healthy),4924,0.3819,0.3729,0.3835,0.6278,0.3403,0.6495,0.3883,0.3818,0.5827,0.6076
FD001,150+ (New),5607,0.3505,0.3461,0.3449,0.6623,0.2975,0.6883,0.3579,0.3547,0.6146,0.6431
FD001,26-50 (Warning),2500,0.5378,0.5063,0.5598,0.4678,0.5259,0.4684,0.5473,0.5164,0.4311,0.4479
FD001,51-100 (Moderate),5000,0.4436,0.4235,0.4504,0.5656,0.4125,0.5813,0.4513,0.4345,0.5227,0.546
FD002,0-25 (Critical),6760,0.4131,0.5005,0.47,0.3486,0.5852,0.3481,0.3835,0.4954,0.361,0.3649
FD002,101-150 (Healthy),12835,0.4035,0.4722,0.4295,0.3487,0.5418,0.3481,0.3567,0.4665,0.3654,0.3692
FD002,150+ (New),14664,0.4036,0.4699,0.4257,0.3487,0.5381,0.348,0.3528,0.4645,0.3658,0.3696
FD002,26-50 (Warning),6500,0.4051,0.4844,0.4477,0.3436,0.5625,0.3431,0.3756,0.479,0.3581,0.3621
FD002,51-100 (Moderate),13000,0.4039,0.4757,0.4358,0.3482,0.5472,0.3476,0.3676,0.4702,0.3643,0.3681
